In [ ]:
import io
from typing import Dict, Tuple
from privacy_preserving_unet_model import PrivacyPreservingUNet
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from scipy.spatial.distance import euclidean
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os

class GazeEstimator:
    def __init__(self, model_path):
        self.camera_params = {
            'focal_length': 800.0,  
            'principal_point': (320.0, 240.0),
            'pixel_size': 0.001, 
        }

        self.eye_params = {
            'eyeball_radius': 12.0,  # mm
            'cornea_radius': 7.8,    # mm
            'iris_radius': 5.9,      # mm (average)
            'pupil_radius': 2.0,     # mm (varies with lighting)
        }

        self.model_path = model_path
        self.model = self.load_model(model_path)

        self.classes = {
            'background': 0,
            'scelar': 1,
            'pupil': 2, 
            'iris': 3,
        }
        self.image_size = (128, 128)  # Default input size for the model

        self.model = self.load_model(model_path)

    def load_model(self, model_path=None):
        # Placeholder for model loading logic
        if model_path is None:
            raise ValueError("Model path must be provided")
        
        self.model_path = model_path

        return PrivacyPreservingUNet.load_model('privacy_preserving_unet.h5')

    def load_image(self ,image_path):
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image file {image_path} does not exist.")
        
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise ValueError("Failed to read the image. Please check the file format.")
        return np.expand_dims(image, axis=(0,-1))


    def calculte_eye_dimensions(self, input_data, output_path=None , index=None):
        if not self.model:
            raise ValueError("Model is not loaded. Please load the model first.")
        predicted_mask = self.model.predict(input_data)

        predicted_mask = np.argmax(predicted_mask, axis=-1)[0]

        # Apply the color mapping
        colored_mask = self.apply_color_map(predicted_mask)

        cv2.imwrite(f'{output_path}/predicted_masks/{index}.png', colored_mask)

        sclera_contours = self.find_sclera_contour(predicted_mask)
        pupil_contour = self.find_pupil_contour(predicted_mask)
        iris_contour = self.find_iris_contour(predicted_mask)
        eye_contour = self.find_eye_contours(predicted_mask)

        if sclera_contours is None or pupil_contour is None or iris_contour is None:
            return None

        pupil_centre, pupil_radius = self._get_contour_properties(pupil_contour)
        iris_centre, iris_radius = self._get_contour_properties(iris_contour)
        sclera_center, sclera_extent = self._get_scelara_properties(sclera_contours)

        # ax_img, ax_mask = self.visualize_segmentation(
        #     input_data,
        #     predicted_mask,
        #     sclera_contours,
        #     pupil_contour,
        #     iris_contour,
        #     pupil_centre,
        #     pupil_radius,
        #     iris_centre,
        #     iris_radius,
        #     sclera_center,
        #     sclera_extent
        # )

        self.draw_on_raw_image(input_data, 
            sclera_center, 
            pupil_centre, 
            iris_centre, 
            eye_contour,
            pupil_contour,
            pupil_radius,
            output_path= f'{output_path}/final_images/{index}.png'
        )

        return{
            'predicted_mask': predicted_mask,
            'sclera_contours': sclera_contours,
            'pupil_contour': pupil_contour,
            'iris_contour': iris_contour,
            'pupil_centre': pupil_centre,
            'pupil_radius': pupil_radius,
            'iris_centre': iris_centre,
            'iris_radius': iris_radius,
            'sclera_center': sclera_center,
            'sclera_extent': sclera_extent,
        }
    
    def apply_color_map(self ,mask):
        # Define RGB colors for each class
        color_map = {
            0: [153, 76, 0],   # B , G , R
            1: [0, 0, 255],    # Red
            2: [255, 255, 0],   # Orange (if you have this class)
            3: [255, 0, 255]   # Purple/Pink
        }
        
        # Create colored mask
        h, w = mask.shape
        colored_mask = np.zeros((h, w, 3), dtype=np.uint8)
        
        for class_id, color in color_map.items():
            colored_mask[mask == class_id] = color
        
        return colored_mask
    
    def visualize_segmentation(
    self,
    actual_image,
    predicted_mask,
    sclera_contours,
    pupil_contour,
    iris_contour,
    pupil_centre,
    pupil_radius,
    iris_centre,
    iris_radius,
    sclera_center,
    sclera_extent,
    save_path=None,  # Add optional save path parameter
    show_plot=True   # Add option to show or just save
    ):
        import matplotlib.pyplot as plt
        import numpy as np
        
        # Show actual image (grayscale or color)
        actual_image = np.squeeze(actual_image)
        
        # Create figure with subplots
        fig_img, ax_img = plt.subplots(figsize=(6, 6))
        fig_mask, ax_mask = plt.subplots(figsize=(6, 6))
        
        # Display the actual image
        if len(actual_image.shape) == 2:  # grayscale
            ax_img.imshow(actual_image, cmap='gray')
        else:  # color
            ax_img.imshow(actual_image)
        ax_img.set_title('Actual Image with Annotations')
        ax_img.axis('off')
        
        # Display the predicted mask
        ax_mask.imshow(predicted_mask, cmap='gray')
        ax_mask.set_title('Predicted Mask with Contours')
        ax_mask.axis('off')
        
        # Scale factors for coordinate transformation
        sclera_center_img_x = (float(sclera_center[0]) / 128.0) * 640
        sclera_center_img_y = (float(sclera_center[1]) / 128.0) * 400
        
        iris_centre_img_x = (float(iris_centre[0]) / 128.0) * 640
        iris_centre_img_y = (float(iris_centre[1]) / 128.0) * 400
        
        pupil_centre_img_x = (float(pupil_centre[0]) / 128.0) * 640
        pupil_centre_img_y = (float(pupil_centre[1]) / 128.0) * 400
        
        # Plot contours if they exist
        if sclera_contours is not None:
            print("Sclera center is :", sclera_center)
            # Plot on mask (original coordinates)
            ax_mask.plot(sclera_center[0], sclera_center[1], 'yo', markersize=8, label='Sclera Center')
            # Plot on actual image (scaled coordinates)
            ax_img.plot(sclera_center_img_x, sclera_center_img_y, 'yo', markersize=8, label='Sclera Center')
            
            # Add sclera bounding box on mask
            left = sclera_center[0] - sclera_extent
            top = sclera_center[1] - sclera_extent
            rect = plt.Rectangle((left, top), 2*sclera_extent, 2*sclera_extent, 
                            linewidth=2, edgecolor='orange', facecolor='none', 
                            linestyle=':', label='Sclera Box')
            ax_mask.add_patch(rect)
        
        if pupil_contour is not None:
            # Plot pupil center on both images
            ax_mask.plot(pupil_centre[0], pupil_centre[1], 'bo', markersize=6, label='Pupil Center')
            ax_img.plot(pupil_centre_img_x, pupil_centre_img_y, 'bo', markersize=6, label='Pupil Center')
            
            # Add pupil circle on mask
            pupil_circle = plt.Circle(pupil_centre, pupil_radius, color='blue', fill=False, 
                                    linestyle='--', linewidth=1)
            ax_mask.add_patch(pupil_circle)
        
        if iris_contour is not None:
            # Plot iris center on both images
            ax_mask.plot(iris_centre[0], iris_centre[1], 'go', markersize=6, label='Iris Center')
            ax_img.plot(iris_centre_img_x, iris_centre_img_y, 'go', markersize=6, label='Iris Center')
            
            # Add iris circle on mask
            iris_circle = plt.Circle(iris_centre, iris_radius, color='green', fill=False, 
                                linestyle='--', linewidth=1)
            ax_mask.add_patch(iris_circle)
        
        # Draw arrows from sclera_center to iris_centre_img and pupil_centre_img on actual image
        if sclera_contours is not None and iris_contour is not None:
            # Arrow from sclera center to iris center
            ax_img.annotate(
                '',
                xy=(iris_centre_img_x, iris_centre_img_y),
                xytext=(sclera_center_img_x, sclera_center_img_y),
                arrowprops=dict(facecolor='red', edgecolor='red', arrowstyle='->', lw=2),
                annotation_clip=False
            )
        
        if sclera_contours is not None and pupil_contour is not None:
            # Arrow from sclera center to pupil center
            ax_img.annotate(
                '',
                xy=(pupil_centre_img_x, pupil_centre_img_y),
                xytext=(sclera_center_img_x, sclera_center_img_y),
                arrowprops=dict(facecolor='magenta', edgecolor='magenta', arrowstyle='->', lw=2),
                annotation_clip=False
            )
        
        # Add legends
        ax_img.legend(loc='upper right')
        ax_mask.legend(loc='upper right')
        
        # Save the figure if save_path is provided
        if save_path:
            plt.tight_layout()  # Adjust layout to prevent clipping
            plt.savefig(save_path, dpi=300, bbox_inches='tight', 
                    facecolor='white', edgecolor='none')
            print(f"Visualization saved to: {save_path}")
        
        # Show the plot if requested
        if show_plot:
            plt.show()

        # Save actual image figure
        fig_img.savefig("./_actual_image.png", dpi=300, bbox_inches='tight')
        # Save predicted mask figure
        fig_mask.savefig("./_mask_image.png", dpi=300, bbox_inches='tight')
                
        return ax_img, ax_mask
    


    def find_sclera_contour(self, predicted_mask):
        # Placeholder for sclera contour finding logic
        sclera_region_mask = (predicted_mask == self.classes['scelar']).astype(np.uint8)
        contours, _ = cv2.findContours(sclera_region_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            largest_contour = [cnt for cnt in contours if cv2.contourArea(cnt) > 25]
            return largest_contour
        return None

    def find_iris_contour(self, predicted_mask):
        pupil_region_mask = (predicted_mask == 2).astype(np.uint8)
        contours, _ = cv2.findContours(pupil_region_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            largest_contour = max(contours, key=cv2.contourArea)
            return largest_contour
        return None    
    
    def find_pupil_contour(self, predicted_mask):
        iris_region_mask = (predicted_mask == 3).astype(np.uint8)
        contours, _ = cv2.findContours(iris_region_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            largest_contour = max(contours, key=cv2.contourArea)
            return largest_contour
        return None
    
    def find_eye_contours(self, predicted_mask):
        eye_region_mask = np.isin(predicted_mask, [1, 2, 3]).astype(np.uint8)
        contours, _ = cv2.findContours(eye_region_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            largest_contour = [cnt for cnt in contours if cv2.contourArea(cnt) > 25]
            return largest_contour
        return None

    
    def _get_contour_properties(self, contour: np.ndarray) -> Tuple[Tuple[int, int], float]:
        (x, y), r = cv2.minEnclosingCircle(contour)
        return (int(round(x)), int(round(y))), float(r)
    
    def _get_scelara_properties(self, contours: np.ndarray) -> Tuple[Tuple[int, int], float]:
        """Calculate sclera center and extent from multiple contours"""
        if not contours:
            return (0, 0), 0
        
        # Combine all sclera points
        all_points = np.vstack([cnt.reshape(-1, 2) for cnt in contours])
        
        # Find extremes
        leftmost = tuple(all_points[np.argmin(all_points[:, 0])])
        rightmost = tuple(all_points[np.argmax(all_points[:, 0])])
        topmost = tuple(all_points[np.argmin(all_points[:, 1])])
        bottommost = tuple(all_points[np.argmax(all_points[:, 1])])
        
        # Calculate center as midpoint of extremes
        center_x = (leftmost[0] + rightmost[0]) // 2
        center_y = (topmost[1] + bottommost[1]) // 2
        
        # Calculate extent as maximum distance from center
        center = np.array([center_x, center_y])
        extent = max([
            np.linalg.norm(center - np.array(leftmost)),
            np.linalg.norm(center - np.array(rightmost)),
            np.linalg.norm(center - np.array(topmost)),
            np.linalg.norm(center - np.array(bottommost))
        ])
        
        return (center_x, center_y), extent
    
    def calculate_3d_gaze_vector(self, iris_center: Tuple[int, int], 
                                sclera_center: Tuple[int, int],
                                iris_radius: float) -> Tuple[np.ndarray, Dict[str, float], float]:
        """
        Calculate 3D gaze vector using geometric eye model
        
        Returns:
            gaze_vector_3d: 3D normalized gaze vector
            gaze_angles: Dictionary with yaw, pitch, roll angles in degrees
            confidence: Confidence score of the calculation
        """
        
        # Convert pixel coordinates to camera coordinates
        fx, fy = self.camera_params['focal_length'], self.camera_params['focal_length']
        cx, cy = self.camera_params['principal_point']
        
        # Iris center in camera coordinates
        iris_x = (iris_center[0] - cx) / fx
        iris_y = (iris_center[1] - cy) / fy
        
        # Sclera center in camera coordinates
        sclera_x = (sclera_center[0] - cx) / fx
        sclera_y = (sclera_center[1] - cy) / fy
        
        # Calculate gaze displacement vector in image plane
        displacement_x = iris_x - sclera_x
        displacement_y = iris_y - sclera_y
        
        # 3D gaze vector calculation using simplified eye model
        # Assume eye is looking straight when iris is centered in sclera
        
        # Scale factors based on anatomical proportions
        scale_x = displacement_x * 2.0  # Empirical scaling
        scale_y = displacement_y * 2.0
        
        # Calculate Z component (depth) using iris size information
        # Smaller iris (farther from camera) suggests looking away
        expected_iris_radius_pixels = self.eye_params['iris_radius'] * fx / 200  # Rough estimate
        z_factor = max(0.1, min(1.0, iris_radius / expected_iris_radius_pixels))
        
        # Construct 3D gaze vector
        gaze_vector_3d = np.array([scale_x, -scale_y, z_factor])  # Negative Y for correct orientation
        
        # Normalize the vector
        gaze_vector_3d = gaze_vector_3d / (np.linalg.norm(gaze_vector_3d) + 1e-8)
        
        # Calculate angles
        yaw = np.degrees(np.arctan2(gaze_vector_3d[0], gaze_vector_3d[2]))
        pitch = np.degrees(np.arcsin(-gaze_vector_3d[1]))
        roll = 0.0  # Cannot determine roll from single eye view
        
        gaze_angles = {
            'yaw': yaw,
            'pitch': pitch,
            'roll': roll
        }
        
        # Calculate confidence based on various factors
        confidence = self._calculate_confidence(iris_center, sclera_center, iris_radius)
        
        return gaze_vector_3d, gaze_angles, confidence

    def _calculate_confidence(self, iris_center: Tuple[int, int], 
                            sclera_center: Tuple[int, int], 
                            iris_radius: float) -> float:
        """Calculate confidence score for gaze estimation"""
        
        confidence_factors = []
        
        # Factor 1: Iris size (reasonable size indicates good detection)
        size_factor = min(1.0, iris_radius / 20.0) if iris_radius > 0 else 0.0
        confidence_factors.append(size_factor)
        
        # Factor 2: Distance between iris and sclera centers (should be reasonable)
        distance = euclidean(iris_center, sclera_center)
        distance_factor = max(0.0, 1.0 - distance / 50.0)  # Penalize if too far apart
        confidence_factors.append(distance_factor)
        
        # Factor 3: Position within image (center is more reliable)
        center_x, center_y = self.image_size[0] // 2, self.image_size[1] // 2
        center_distance = euclidean(iris_center, (center_x, center_y))
        max_distance = np.sqrt(center_x**2 + center_y**2)
        position_factor = max(0.0, 1.0 - center_distance / max_distance)
        confidence_factors.append(position_factor)
        
        # Combined confidence
        confidence = np.mean(confidence_factors)
        return confidence

    
    def draw_contour(self, image: np.ndarray, contour: np.ndarray, color: Tuple[int, int, int], thickness: int = 2) -> np.ndarray:
        """Draw contour on the image"""
        if contour is not None:
            cv2.drawContours(image, [contour], -1, color, thickness)
        return image
    
    def draw_on_raw_image(self, image, sclera_center, pupil_center, iris_center, sclera_contours, pupil_contour,pupil_radius, output_path):
    # Make a copy to avoid modifying the original
        img = image.copy()

        W, H   = 640, 400          # full‑resolution
        w0, h0 = 128, 128          # mask resolution

        scale = np.array([W / w0, H / h0])
        magnitude = np.linalg.norm(scale)

        # your 3 points as a single (3, 2) array  – order:  (x, y)
        coords = np.array([sclera_center,
                        pupil_center,
                        iris_center], dtype=np.float32)

        # multiply once, broadcast takes care of each column
        scaled = (coords * scale).round().astype(int)

        sclera_center_scaled, pupil_center_scaled, iris_center_scaled = scaled

        img = np.squeeze(image).copy()

        if img.dtype != np.uint8:
            img = (img * 255).clip(0, 255).astype(np.uint8)

        # Convert grayscale to BGR if needed
        if len(img.shape) == 2 or img.shape[2] == 1:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

        # Convert float coordinates to integers
        sclera_center = tuple(map(int, sclera_center_scaled))
        pupil_center = tuple(map(int, pupil_center_scaled))
        iris_center = tuple(map(int, iris_center_scaled))

        scaled_contours = [
        (cnt.reshape(-1, 2).astype(np.float32) * scale).round().astype(int)
        .reshape(-1, 1, 2)                          # cv2 expects (N,1,2)
        for cnt in sclera_contours
        ]

        cv2.drawContours(img, scaled_contours, -1, (0, 0, 255), 2)

        # Draw centers
        cv2.circle(img, sclera_center, 6, (0, 255, 255), -1)   # Yellow for Sclera
        cv2.circle(img, pupil_center, 5, (255, 0, 0), -1)      # Blue for Pupil
        cv2.circle(img, iris_center, 5, (0, 255, 0), -1)       # Green for Iris
        cv2.circle(img, pupil_center, round(pupil_radius* magnitude/2), (255, 255, 0), thickness= 2)

        # Draw arrows from sclera center to pupil and iris
        cv2.arrowedLine(img, sclera_center, pupil_center, (255, 0, 255), 2, tipLength=0.3)  # Magenta
        # Save output
        cv2.imwrite(output_path, img)

def trailing_number(path: str) -> int:
    fname = os.path.basename(path)          # "0.png"  -> 0
    stem  = os.path.splitext(fname)[0]      # "0"      -> still "0"
    return int(stem)   

def images_to_video(folder_path, output_path='output.mp4', fps=2):
    """
    Convert image sequence to video
    """
    image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
    image_files = [f for f in os.listdir(folder_path) 
                   if f.lower().endswith(image_extensions)]
    
                         # convert to int

    # `image_paths` is your unsorted list of full paths
    sorted_paths = sorted(image_files, key=trailing_number)
    
    if not sorted_paths:
        print("No images found!")
        return
    
    # Get image dimensions
    first_image = cv2.imread(os.path.join(folder_path, sorted_paths[0]))
    height, width, layers = first_image.shape
    
    # Create video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    for image_file in sorted_paths:
        image_path = os.path.join(folder_path, image_file)
        img = cv2.imread(image_path)
        video.write(img)
    
    video.release()
    print(f"Video saved as {output_path}")

In [8]:
import glob
import os

gaze_estimator = GazeEstimator(model_path='privacy_preserving_unet_medium.h5')

# Get all folder names inside the dataset directory
dataset_dir = "/home/yasas/GazeEstimation/openEDS/openEDS"
person_folders = [f for f in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, f))]
print("Found folders:", person_folders)

for person in person_folders:
    if person == 'train' or person == 'validation' or person == 'test':
        continue
    print(f"Processing person: {person}")
    image_paths = sorted(glob.glob(f'/home/yasas/GazeEstimation/openEDS/openEDS/{person}/*.png'))

    def trailing_number(path: str) -> int:
        fname = os.path.basename(path)          # "0.png"  -> 0
        stem  = os.path.splitext(fname)[0]      # "0"      -> still "0"
        return int(stem)                        # convert to int

    # `image_paths` is your unsorted list of full paths
    sorted_paths = sorted(image_paths, key=trailing_number)
    person = sorted_paths[0].split('/')[-2]  # Extract person name from the first path
    print(person)

    for img_path in sorted_paths:
        input_data = gaze_estimator.load_image(img_path)
        if not os.path.exists(f"./output_images/{person}"):
            os.makedirs(f"./output_images/{person}")
            os.makedirs(f"./output_images/{person}/predicted_masks")
            os.makedirs(f"./output_images/{person}/final_images")
        result = gaze_estimator.calculte_eye_dimensions(input_data, f"./output_images/{person}", trailing_number(img_path)) 
    output_path = f"./output_video/{person}"
    image_sequence_path = f'/home/yasas/GazeEstimation/output_images/{person}/final_images'
    predicted_mask_sequence_path = f'/home/yasas/GazeEstimation/output_images/{person}/predicted_masks'
    if not os.path.exists(output_path):
        os.makedirs(output_path)
        os.makedirs(output_path + '/final_video')
    if not os.path.exists(output_path + '/predicted_masks_video'):
        os.makedirs(output_path + '/predicted_masks_video')
        os.makedirs(output_path + '/raw_video')
    images_to_video(image_sequence_path, output_path + '/final_video' + f'/{person}_final.mp4', fps=15)
    images_to_video(predicted_mask_sequence_path, output_path+ '/predicted_masks_video' + f'/{person}_predicted.mp4', fps=15)
    images_to_video(f'/home/yasas/GazeEstimation/openEDS/openEDS/{person}', output_path+ '/raw_video' + f'/{person}_actual.mp4', fps=15)
    break

Found folders: ['S_173', 'S_14', 'S_72', 'S_12', 'S_130', 'train', 'S_115', 'S_171', 'S_182', 'S_62', 'S_160', 'S_159', 'S_57', 'S_178', 'S_192', 'S_34', 'S_122', 'S_58', 'validation', 'S_13', 'S_41', 'S_32', 'S_99', 'S_117', 'S_140', 'S_111', 'S_165', 'S_29', 'S_7', 'S_151', 'S_63', 'S_104', 'S_38', 'S_162', 'S_158', 'S_28', 'S_91', 'S_124', 'S_191', 'S_148', 'S_15', 'S_8', 'S_50', 'test', 'S_144', 'S_37', 'S_66', 'S_197', 'S_31', 'S_70', 'S_20', 'S_23', 'S_60', 'S_175', 'S_6', 'S_30', 'S_167', 'S_93', 'S_195', 'S_84', 'S_2', 'S_156', 'S_164', 'S_177', 'S_194', 'S_51', 'S_81', 'S_55', 'S_183', 'S_179', 'S_143', 'S_45', 'S_110', 'S_0', 'S_189', 'S_116', 'S_102', 'S_135', 'S_97', 'S_43', 'S_126', 'S_73', 'S_94', 'S_108', 'S_145', 'S_9', 'S_141', 'S_129', 'S_186', 'S_139', 'S_42', 'S_24', 'S_76', 'S_187', 'S_27', 'S_106', 'S_125', 'S_114', 'S_47', 'S_154', 'S_113', 'S_46', 'S_107', 'S_138', 'S_89', 'S_109', 'S_40', 'S_147', 'S_155', 'S_103', 'S_146', 'S_149', 'S_54', 'S_33', 'S_188', 'S_